In [1]:
#see https://www.pinecone.io/learn/series/nlp/fine-tune-sentence-transformers-mnr/
import datasets

# snli = datasets.load_dataset('snli', split='train')
# mnli = datasets.load_dataset('glue', 'mnli', split='train')

# mnli=mnli.remove_columns("idx")

# snli = snli.cast(mnli.features)

# dataset = datasets.concatenate_datasets([snli, mnli])

# del snli, mnli

# print(f"before: {len(dataset)} rows")
# dataset = dataset.filter(
#     lambda x: True if x['label'] == 0 else False
# )
# print(f"after: {len(dataset)} rows")



/home/kperkins411/anaconda3/envs/p311/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [15]:
from datasets import load_dataset, concatenate_datasets
test_dataset = load_dataset("json", data_files="tst.json", split="train")
train_dataset = load_dataset("json", data_files="trn.json", split="train")
corpus_dataset = concatenate_datasets([train_dataset, test_dataset])

# Convert the datasets to dictionaries
corpus = dict(
    zip(corpus_dataset["id"], corpus_dataset["positive"])
)  # Our corpus (cid => document)
queries = dict(
    zip(test_dataset["id"], test_dataset["anchor"])
)  

# Create a mapping of relevant document (1 in our case) for each query
relevant_docs = {}  # Query ID to relevant documents (qid => set([relevant_cids])
for q_id in queries:
    relevant_docs[q_id] = [q_id]
 
print(f"{len(train_dataset)} rows")
print(train_dataset)
print(train_dataset[0])

31837 rows
Dataset({
    features: ['id', 'most_dissimilar_context', 'anchor', 'positive'],
    num_rows: 31837
})
{'id': 0, 'most_dissimilar_context': 'If such Standard Cost methodology change results in an increase of Facility Conversion Cost for Products manufactured for Customer of more than two percent (2%), then Manufacturer shall revert to the former methodology for purposes of the calculation of Price during such Fiscal Year.', 'anchor': 'What safeguards are in place to protect the information obtained from third-party sources?', 'positive': 'Information We Collect From Other Sources We may also receive information from other sources and combine that with information we collect through our Services. For example: If you choose to link, create, or log in to your Uber account with a payment provider (e.g., Google Wallet) or social media service (e.g., Facebook), or if you engage with a separate app or website that uses our API (or whose API we use), we may receive information abou

In [3]:
print(train_dataset[0])

{'id': 0, 'most_dissimilar_context': 'If such Standard Cost methodology change results in an increase of Facility Conversion Cost for Products manufactured for Customer of more than two percent (2%), then Manufacturer shall revert to the former methodology for purposes of the calculation of Price during such Fiscal Year.', 'anchor': 'What safeguards are in place to protect the information obtained from third-party sources?', 'positive': 'Information We Collect From Other Sources We may also receive information from other sources and combine that with information we collect through our Services. For example: If you choose to link, create, or log in to your Uber account with a payment provider (e.g., Google Wallet) or social media service (e.g., Facebook), or if you engage with a separate app or website that uses our API (or whose API we use), we may receive information about you or your connections from that site or app.'}


In [4]:
from sentence_transformers import InputExample
from tqdm.auto import tqdm  # so we see progress bar

train_samples = []
for row in tqdm(train_dataset):
    train_samples.append(InputExample(
        texts=[row['positive'], row['anchor']]
    ))

100%|██████████| 31837/31837 [00:00<00:00, 33689.42it/s]


In [5]:
from sentence_transformers import datasets

batch_size = 32

loader = datasets.NoDuplicatesDataLoader(
    train_samples, batch_size=batch_size)

In [6]:
from sentence_transformers import models, SentenceTransformer

bert = models.Transformer('bert-base-uncased')
pooler = models.Pooling(
    bert.get_word_embedding_dimension(),
    pooling_mode_mean_tokens=True
)

model = SentenceTransformer(modules=[bert, pooler])

model

/home/kperkins411/anaconda3/envs/p311/lib/python3.11/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


SentenceTransformer(
  (0): Transformer({'max_seq_length': 512, 'do_lower_case': False}) with Transformer model: BertModel 
  (1): Pooling({'word_embedding_dimension': 768, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
)

In [8]:
from sentence_transformers import losses

loss = losses.MultipleNegativesRankingLoss(model)

In [10]:
#liit to 1 GPU
import torch
device = torch.device("cuda:1") # to use GPU ID 1 only
# trainer.args._n_gpu = 1

In [11]:
epochs = 1
warmup_steps = int(len(loader) * epochs * 0.1)

model.fit(
    train_objectives=[(loader, loss)],
    epochs=epochs,
    warmup_steps=warmup_steps,
    output_path='./sbert_test_mnr2',
    show_progress_bar=True
)  # I set 'show_progress_bar=False' as it printed every step
#    on to a new line

Currently using DataParallel (DP) for multi-gpu training, while DistributedDataParallel (DDP) is recommended for faster training. See https://sbert.net/docs/sentence_transformer/training/distributed.html for more information.


Step,Training Loss
500,0.766300


In [16]:
from sentence_transformers.evaluation import (
    InformationRetrievalEvaluator,
    SequentialEvaluator,
)
dev_evaluator = InformationRetrievalEvaluator(
    queries=queries,
    corpus=corpus,
    relevant_docs=relevant_docs,
    name=f"delme",
)
print(dev_evaluator(model))

{'delme_cosine_accuracy@1': 0.041965764770844835, 'delme_cosine_accuracy@3': 0.1424627277747101, 'delme_cosine_accuracy@5': 0.2390944229707344, 'delme_cosine_accuracy@10': 0.44450579790171174, 'delme_cosine_precision@1': 0.041965764770844835, 'delme_cosine_precision@3': 0.04748757592490336, 'delme_cosine_precision@5': 0.04781888459414688, 'delme_cosine_precision@10': 0.044450579790171175, 'delme_cosine_recall@1': 0.041965764770844835, 'delme_cosine_recall@3': 0.1424627277747101, 'delme_cosine_recall@5': 0.2390944229707344, 'delme_cosine_recall@10': 0.44450579790171174, 'delme_cosine_ndcg@10': 0.2046154994319853, 'delme_cosine_mrr@10': 0.13298230390996785, 'delme_cosine_map@100': 0.1499048526434139, 'delme_dot_accuracy@1': 0.041413583655438985, 'delme_dot_accuracy@3': 0.13086692435118719, 'delme_dot_accuracy@5': 0.2098288238542242, 'delme_dot_accuracy@10': 0.390944229707344, 'delme_dot_precision@1': 0.041413583655438985, 'delme_dot_precision@3': 0.04362230811706239, 'delme_dot_precision